# IHDP-100 initial inspection — Phase 2

Objective: inspect the exact author-hosted train/test release without changing its values, fitting transformations or performing EDA. Run the downloader from the repository root first. Source evidence is in `docs/references.md`; hashes are locally recorded baselines, not publisher signatures. Keep row-level outputs local while redistribution terms remain unresolved.

This notebook calls reusable package logic. Replication indices are zero-based. No row alignment across replications is assumed.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
from causallens.data.loader import load_ihdp, load_ihdp_truth, archive_path
from causallens.data.inspection import inspect_ihdp
root = Path.cwd()
if not (root / "pyproject.toml").exists():
    root = root.parent
assert (root / "pyproject.toml").exists(), "Run from repository root or notebooks/"
data_dir = root / "data/raw"
replication = 0
print("Python:", sys.version.split()[0], "NumPy:", np.__version__, "pandas:", pd.__version__)
result = inspect_ihdp(data_dir, replication)


## Archive schema
CRC and structural checks inspect all stored replications. The final axis selects the replication; scalar metadata is separate.

In [ ]:
schema = []
for split, info in result["splits"].items():
    for key, a in info["schema"].items():
        schema.append({"split": split, "key": key, **a})
display(pd.DataFrame(schema))


## Replication 0 counts and factual outcome summary
These are requested initial inspection summaries, not causal estimates or model tuning.

In [ ]:
summary = [{"split": s, "n": v["n"], "features": v["n_features"],
            "treated": v["treated"], "control": v["control"], **v["yf"]}
           for s,v in result["splits"].items()]
display(pd.DataFrame(summary))


## Small observed-data tables (local outputs only)
Five rows with three generic covariates per split. These are source values; do not infer clinical units.

In [ ]:
for split, info in result["splits"].items():
    print(split)
    display(pd.DataFrame(info["sample_observed_rows"]))


## Empirical feature encodings
Confirm the six multi-valued and nineteen two-level columns. In this release x14 uses {1,2}; do not silently recode it or infer a clinical meaning.

In [ ]:
feature_checks = [{"split":split, **f} for split, info in result["splits"].items()
                  for f in info["features"]]
display(pd.DataFrame(feature_checks))


## Model versus evaluation boundary
Observed loader returns X, T, Y only. Oracle arrays require a separate accessor. No causal effect estimates or preprocessing are computed here.

In [ ]:
observed = load_ihdp(replication, "train", data_dir)
truth = load_ihdp_truth(replication, "train", data_dir)
assert set(observed) == {"X", "T", "Y"}
assert set(truth) == {"mu0", "mu1", "ycf", "tau"}
print("Observed:", {k:v.shape for k,v in observed.items()})
print("Evaluation only:", {k:v.shape for k,v in truth.items()})
with np.load(archive_path("train",data_dir), allow_pickle=False) as source:
    np.testing.assert_array_equal(observed["X"], source["x"][:,:,replication])
    print("Scalar metadata:", {k: source[k].item() for k in ("ate","yadd","ymul")})
# Noise means factual responses are not required to equal their expected means.
print("yf exactly equals assigned expected mean:", result["splits"]["train"]["yf_equals_assigned_expected_mean"])
print("Split fingerprint check:", result["split_check"])


## Interpretation and remaining limitations

- The raw archives are unchanged and official train/test boundaries are retained.
- Exact X+T row fingerprints check apparent duplication, not patient identity; there is no verified ID mapping.
- Structural checks find finite arrays. They do not establish overlap or causal identification.
- Generic feature names remain canonical; detailed coding and dataset redistribution terms are unresolved.
- The scalar `ate` field is not substituted for split-specific mean(mu1−mu0).
- Phases 3 and 4 are implemented in notebooks 02 and 03. Phase 5 full validation remains separate.